# Context Graphs for AI Agents: Explorer Path

This notebook is the browser-only path for the workshop.

You will:

1. Paste your NAMS credentials.
2. Store two decisions in two separate conversations.
3. Watch Neo4j Agent Memory extract entities and relationships.
4. Query the graph for the right memory.

You do not need VS Code, git, local Python, or Neo4j Desktop. The notebook calls the real NAMS REST API.

## 1. Paste Your NAMS Credentials

Find these at [memory.neo4jlabs.com](https://memory.neo4jlabs.com) under **REST → Make your first call**.

Do not paste your API key in chat or on a screenshare.

In [1]:
from getpass import getpass

MEMORY_API_KEY = getpass("Paste MEMORY_API_KEY (input is hidden): ").strip()
NAMS_WORKSPACE_ID = input("Paste NAMS_WORKSPACE_ID: ").strip()

assert MEMORY_API_KEY.startswith("nams_"), "Your API key should start with nams_"
assert NAMS_WORKSPACE_ID, "Workspace id is required"

print("Credentials loaded for this notebook session.")

Paste MEMORY_API_KEY (input is hidden): ··········
Paste NAMS_WORKSPACE_ID: 7163473d-9bcc-4b9a-bfae-15723557c1fe
Credentials loaded for this notebook session.


## 2. Choose Your Story

The default story uses the June 11th continuity example: Zustand vs. Redux.

To make your own graph, edit the two decision sentences below. Keep one repeated phrase across both sessions, such as a project, product, customer, trip, or team. That shared phrase gives the graph something to connect.

In [2]:
#@title Story fields
RESET_WORKSPACE_FIRST = True #@param {type:"boolean"}
SESSION_1_DECISION = "On the Context Graph Workshop project, we chose Zustand for cart state. We considered Redux but rejected it because of re-render performance problems." #@param {type:"string"}
SESSION_2_DECISION = "On the Context Graph Workshop project, we chose Neo4j to store agent memory. We considered Postgres but rejected it because we needed relationship-first queries." #@param {type:"string"}
RECALL_QUERY = "what did we decide for cart state and what did we reject?" #@param {type:"string"}

DECISIONS = [SESSION_1_DECISION, SESSION_2_DECISION]

print("Story loaded:")
for i, decision in enumerate(DECISIONS, start=1):
    print(f"Session {i}: {decision}")
print(f"Query: {RECALL_QUERY}")

Story loaded:
Session 1: On the Context Graph Workshop project, we chose Zustand for cart state. We considered Redux but rejected it because of re-render performance problems.
Session 2: On the Context Graph Workshop project, we chose Neo4j to store agent memory. We considered Postgres but rejected it because we needed relationship-first queries.
Query: what did we decide for cart state and what did we reject?


## 3. Helper Code

Run this cell once. You do not need to edit it.

In [3]:
import json
import time
import urllib.request
import urllib.error

BASE = "https://memory.neo4jlabs.com/v1"

def call(method, path, body=None):
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(BASE + path, data=data, method=method)
    req.add_header("Authorization", f"Bearer {MEMORY_API_KEY}")
    req.add_header("X-Workspace-Id", NAMS_WORKSPACE_ID)
    if data is not None:
        req.add_header("Content-Type", "application/json")
    try:
        with urllib.request.urlopen(req) as resp:
            raw = resp.read().decode("utf-8")
            return json.loads(raw or "{}")
    except urllib.error.HTTPError as err:
        detail = err.read().decode("utf-8")
        raise RuntimeError(f"NAMS API error {err.code}: {detail}") from err

def list_entities():
    return call("GET", "/entities?limit=100").get("entities", [])

def list_conversations():
    return call("GET", "/conversations").get("conversations", [])

def reset_workspace():
    entities = list_entities()
    for entity in entities:
        call("DELETE", f"/entities/{entity['id']}")
    conversations = list_conversations()
    for conversation in conversations:
        try:
            call("DELETE", f"/conversations/{conversation['id']}")
        except Exception:
            pass
    print(f"Reset complete: deleted {len(entities)} entities and {len(conversations)} conversations.")

def wait_for_extraction(conversation_id, baseline_count, max_seconds=210):
    previous_count = baseline_count
    stable_polls = 0
    started = time.time()
    while time.time() - started < max_seconds:
        time.sleep(5)
        print(".", end="", flush=True)
        count = len(list_entities())
        status = json.dumps(call("GET", f"/conversations/{conversation_id}/extraction-status"))
        grew = count > baseline_count
        stable_polls = stable_polls + 1 if grew and count == previous_count else 0
        if grew and ('"status":"success"' in status or stable_polls >= 1):
            print(" done")
            return
        previous_count = count
    print(" still processing")

def store_decision(message, session_number):
    before = len(list_entities())
    conversation = call("POST", "/conversations", {"userId": "colab-explorer"})
    conversation_id = conversation["id"]
    call("POST", f"/conversations/{conversation_id}/messages", {"role": "user", "content": message})
    print(f"Session {session_number}: stored message; extracting", end="", flush=True)
    wait_for_extraction(conversation_id, before)
    after = len(list_entities())
    print(f"Graph size: {before} -> {after} entities")
    return before, after

def print_entities():
    print("\nEntities in the knowledge graph:")
    for entity in list_entities():
        print(f"- {entity['name']} ({entity['type']})")

def print_graph():
    graph = call("GET", "/entities/graph")
    names = {node["id"]: node.get("name", node["id"][:8]) for node in graph.get("nodes", [])}
    degree = {}
    print("\nRelationships in the context graph:")
    edges = graph.get("edges", [])
    if not edges:
        print("No relationships yet. Relationship extraction can lag behind entity extraction.")
        return
    for edge in edges:
        source = names.get(edge["sourceId"], edge["sourceId"][:8])
        target = names.get(edge["targetId"], edge["targetId"][:8])
        print(f"- ({source}) -[{edge['type']}]-> ({target})")
        degree[source] = degree.get(source, 0) + 1
        degree[target] = degree.get(target, 0) + 1
    if degree:
        top_degree = max(degree.values())
        hubs = [name for name, value in degree.items() if value == top_degree]
        print(f"\nLikely hub connecting memories: {', '.join(hubs)} ({top_degree} links)")

def query_graph(query):
    print(f"\nMemory query: {query}")
    results = call("POST", "/entities/search", {"query": query}).get("entities", [])[:5]
    if not results:
        print("No results yet. Wait a minute and rerun this cell.")
    for result in results:
        description = result.get("description", "")[:80]
        print(f"- {result.get('score', 0):.2f} {result['name']} ({result['type']}): {description}")

print("Helper functions loaded.")

Helper functions loaded.


## 4. Run The Hack

This cell creates two conversations, one for each decision. The graph should grow after each session.

In [4]:
if RESET_WORKSPACE_FIRST:
    reset_workspace()

sizes = [len(list_entities())]
print(f"Starting graph size: {sizes[0]} entities\n")

for index, decision in enumerate(DECISIONS, start=1):
    _, after = store_decision(decision, index)
    sizes.append(after)
    print()

print(f"Knowledge graph grew across {len(DECISIONS)} sessions: {sizes[0]} -> {sizes[-1]} entities")
print_entities()
print_graph()
query_graph(RECALL_QUERY)

print("\nDemo sentence:")
print(f"My graph grew across {len(DECISIONS)} sessions from {sizes[0]} to {sizes[-1]} entities, and the memory query returned the relevant context.")

Reset complete: deleted 8 entities and 2 conversations.
Starting graph size: 0 entities

Session 1: stored message; extracting..... done
Graph size: 0 -> 6 entities

Session 2: stored message; extracting..... done
Graph size: 6 -> 11 entities

Knowledge graph grew across 2 sessions: 0 -> 11 entities

Entities in the knowledge graph:
- Reject Postgres (Decision)
- agent memory (Concept)
- relationship-first queries (Concept)
- Postgres (Database)
- Neo4j (Database)
- Context Graph Workshop (project)
- Reject Redux (Decision)
- cart state (Concept)
- re-render performance problems (Concept)
- Zustand (SoftwareTool)
- Redux (SoftwareTool)

Relationships in the context graph:
- (Redux) -[RELATED_TO]-> (re-render performance problems)
- (Context Graph Workshop) -[RELATED_TO]-> (Reject Redux)
- (Context Graph Workshop) -[RELATED_TO]-> (Zustand)
- (Context Graph Workshop) -[RELATED_TO]-> (Neo4j)
- (Context Graph Workshop) -[RELATED_TO]-> (Reject Postgres)
- (Reject Redux) -[RELATED_TO]-> (Red

## 5. What To Demo

Fill in this sentence for chat or backstage:

```text
My graph grew from __ entities to __ entities across two sessions.
The two sessions were connected through ______.
NAMS extracted these decisions/rejections: ______.
I asked "______" and the graph returned ______.
```

Engineer stretch: turn off `RESET_WORKSPACE_FIRST`, add a third decision, rerun, and inspect how the graph keeps growing.